# Semana 4 — A Ponte para Tensors: PyTorch e Autograd
**Fase 1: Math Foundations e ML Core** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/01-Phase-1-Math-and-ML-Core/04-pytorch-tensors/notebooks/semana_04_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-04-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/01-Phase-1-Math-and-ML-Core/04-pytorch-tensors/docs/Semana-04-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_04_gabarito.ipynb`.

> **GPU:** em *Ambiente de execução → Alterar o tipo de ambiente de execução*, selecione **T4 GPU** antes de começar.

## Passo 1 — Ambiente

In [ ]:
import torch, numpy as np, time, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "sem GPU")
torch.manual_seed(42)

## Passo 2 — Tensores: shape, dtype, device

In [ ]:
a_np = np.arange(6, dtype=np.float32).reshape(2, 3)
# [SEU CÓDIGO AQUI] crie t a partir de a_np (torch.from_numpy) e t_gpu movendo t para device
print(t.shape, t.dtype, t.device, "->", t_gpu.device)

a_np[0, 0] = 99          # from_numpy compartilha memória na CPU
print("t enxerga a mudança no array?", t[0, 0].item() == 99)

## Passo 3 — Autograd confere a Semana 3
Recrie os dados `y = 2x + 1 + ruído`. Com `w=0.3` e `b=-0.7`, calcule o MSE e chame `.backward()`. Compare `w.grad` e `b.grad` com as fórmulas analíticas da Semana 3.

In [ ]:
x = torch.empty(200).uniform_(-3, 3)
y = 2 * x + 1 + 0.5 * torch.randn(200)

w = torch.tensor(0.3, requires_grad=True)
b = torch.tensor(-0.7, requires_grad=True)
# [SEU CÓDIGO AQUI] calcule loss (MSE) e chame loss.backward()

erro = (w * x + b - y).detach()
dw_manual, db_manual = 2 * (erro * x).mean(), 2 * erro.mean()
print(f"autograd: dw={w.grad:.5f} db={b.grad:.5f}\nmanual  : dw={dw_manual:.5f} db={db_manual:.5f}")
assert torch.allclose(w.grad, dw_manual, atol=1e-5) and torch.allclose(b.grad, db_manual, atol=1e-5)

## Passo 4 — Gradientes acumulam
Chame `backward()` de novo sem zerar e veja `w.grad` dobrar. Depois zere.

In [ ]:
g1 = w.grad.clone()
((w * x + b - y) ** 2).mean().backward()
print("acumulou?", torch.allclose(w.grad, 2 * g1))
# [SEU CÓDIGO AQUI] zere os gradientes de w e b
assert w.grad.item() == 0

## Passo 5 — Loop manual com tensores (ainda sem `nn`/`optim`)
Atualize os parâmetros dentro de `torch.no_grad()` e zere os gradientes a cada epoch.

In [ ]:
w = torch.zeros((), requires_grad=True); b = torch.zeros((), requires_grad=True)
lr = 0.05
for epoca in range(100):
    loss = ((w * x + b - y) ** 2).mean()
    # [SEU CÓDIGO AQUI] backward, atualização sob no_grad, zerar gradientes
print(f"w={w.item():.3f} b={b.item():.3f} loss={loss.item():.4f}")

## Passo 6 — A versão idiomática: `nn.Linear` + `torch.optim`
`nn.Linear` espera entrada com shape `(N, 1)`. Mova modelo e dados para `device` e escreva os cinco passos do loop canônico.

In [ ]:
X = x.unsqueeze(1).to(device); Y = y.unsqueeze(1).to(device)
# [SEU CÓDIGO AQUI] defina modelo, criterio (MSELoss) e otimizador (SGD, lr=0.05)

historico = []
for epoca in range(100):
    # [SEU CÓDIGO AQUI] os cinco passos: forward, loss, zero_grad, backward, step
    historico.append(loss.item())

print(f"w={modelo.weight.item():.3f} b={modelo.bias.item():.3f} loss={historico[-1]:.4f}")
plt.plot(historico); plt.yscale("log"); plt.xlabel("época"); plt.ylabel("MSE"); plt.show()

## Passo 7 — Avaliação sem graph

In [ ]:
# [SEU CÓDIGO AQUI] dentro de torch.no_grad(), preveja para x = 4.0
print("f(4) ≈", round(pred.item(), 3), "(esperado ≈ 9) | requires_grad:", pred.requires_grad)

## Passo 8 — CPU vs. GPU
Multiplique duas matrizes `n × n` em cada device. Lembre do `torch.cuda.synchronize()`.

In [ ]:
def cronometrar(n, dev, reps=3):
    A = torch.randn(n, n, device=dev); B = torch.randn(n, n, device=dev)
    _ = A @ B                                   # aquecimento
    if dev == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        _ = A @ B
    # [SEU CÓDIGO AQUI] sincronize a GPU antes de ler o relógio
    return (time.perf_counter() - t0) / reps * 1000

for n in (64, 512, 2048):
    linha = f"n={n:5d} | CPU {cronometrar(n, 'cpu'):8.2f} ms"
    if device == "cuda": linha += f" | GPU {cronometrar(n, 'cuda'):8.2f} ms"
    print(linha)

## Passo 9 — Memória de GPU

In [ ]:
if device == "cuda":
    grande = torch.randn(8192, 8192, device="cuda")
    print(f"alocado: {torch.cuda.memory_allocated()/1e6:.0f} MB")
    del grande; torch.cuda.empty_cache()
    print(f"após liberar: {torch.cuda.memory_allocated()/1e6:.0f} MB")
else:
    print("Sem GPU neste ambiente: ative a T4 no Colab para ver esta célula.")

## Fechamento da Fase 1
Registre em `docs/` os tempos CPU vs. GPU e responda: a partir de que tamanho a GPU compensa? Faça o commit final da fase.